In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.naive_bayes import GaussianNB

In [7]:
data = pd.read_csv('../data/drugY.csv')

In [10]:
X = data.drop('Drug', axis=1)

In [11]:
y = data['Drug'] * 2 - 1

In [12]:
X = pd.get_dummies(X)

In [13]:
X.head()

,Age,Na,K,Drug_binary,Sex_F,Sex_M,BP_HIGH,BP_LOW,BP_NORMAL,Cholesterol_HIGH,Cholesterol_NORMAL
0,23,0.792535,0.031258,0,True,False,True,False,False,True,False
1,47,0.739309,0.056468,0,False,True,False,True,False,True,False
2,47,0.697269,0.068944,0,False,True,False,True,False,True,False
3,28,0.563682,0.072289,0,True,False,False,False,True,True,False
4,61,0.559294,0.030998,0,True,False,False,True,False,True,False


In [15]:
n = len(X)
# Inicijalizacija jednakih težina primera
instance_weights = pd.Series(np.array([1 / n] * n), index=X.index)
instance_weights.head()

0    0.005
1    0.005
2    0.005
3    0.005
4    0.005
dtype: float64

In [17]:
instance_weights.iloc[0]

np.float64(0.005)

In [16]:
instance_weights.sum()

np.float64(0.9999999999999998)

In [19]:
ensemble = []
model_weights = []

In [20]:
# 1. Kreiranje i treniranje modela
alg = GaussianNB()
model = alg.fit(X, y, sample_weight=instance_weights)

# 2. Predikcija i računanje greške
predictions = model.predict(X)
error = (predictions != y).astype(int)
weighted_error = (error * instance_weights).sum()

# 3. Računanje alfe (learning_rate = 1.0)
alfa = 1.0 * 0.5 * math.log((1 - weighted_error) / weighted_error)

# Čuvanje rezultata
ensemble.append(model)
model_weights.append(alfa)

# 4. Ažuriranje težina primera
factor = np.exp(-alfa * predictions * y)
instance_weights = instance_weights * factor
instance_weights = instance_weights / instance_weights.sum()

# --- ISPIS ZA ANALIZU ---
print(f"--- REZULTATI OVE ITERACIJE ---")
print(f"Broj grešaka modela: {error.sum()} od ukupno {n}")
print(f"Ponderisana greška (weighted_error): {weighted_error:.4f}")
print(f"Težina modela (alfa): {alfa:.4f}")
print(f"Nakon ažuriranja, težine se i dalje sabiraju na: {instance_weights.sum():.1f}")
print("\nStatistika novih težina primera:")
print(f" - Maksimalna težina (težak primer): {instance_weights.max():.4f}")
print(f" - Minimalna težina (lak primer): {instance_weights.min():.4f}")


--- REZULTATI OVE ITERACIJE ---
Broj grešaka modela: 19 od ukupno 200
Ponderisana greška (weighted_error): 0.0950
Težina modela (alfa): 1.1270
Nakon ažuriranja, težine se i dalje sabiraju na: 1.0

Statistika novih težina primera:
 - Maksimalna težina (težak primer): 0.0263
 - Minimalna težina (lak primer): 0.0028
